# 07. Reservoir Computing: Echo State Networks, LSM, and NG-RC

- **Category:** 05_architectures / 03_recurrent_ssm_and_memory
- **Lineage & Citations:** Herbert Jaeger, 2001 (*Echo State Networks*); Wolfgang Maass et al., 2002 (*Liquid State Machines*); Daniel Gauthier et al., 2021 (*Next Generation Reservoir Computing*).
- **Purpose:** Maps inputs into a high-dimensional fixed dynamical reservoir, eliminating Backpropagation Through Time (BPTT) and training only a linear readout in milliseconds via closed-form ridge regression.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Memory Architectures NTM PKM PEER Infini and Titans](06_memory_architectures_ntm_pkm_peer_infini_and_titans.ipynb) $\to$ `🔗 Downstream Syntheses:` [Attention Variants MHA MQA GQA MLA and DiffAttn](../04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **Deep Multimodal Representation Learning:** Reservoirs do not tune internal representations; modern deep Transformers excel where end-to-end representation adaptation is needed.
> - **Unchecked Spectral Radius:** Setting $\rho(W_{\text{res}}) \ge 1.0$ destroys fading memory and triggers self-sustaining chaotic divergence.

## Mathematical Foundations & Governing Equations

### 1. Leaky Echo State Network (ESN) Dynamics
Given fixed reservoir weights $W_{\text{res}}$ with spectral radius $\rho(W_{\text{res}}) < 1$ and leak rate $\alpha \in (0, 1]$:
$$
\tilde{h}_t = \tanh(x_t W_{\text{in}}^\top + h_{t-1} W_{\text{res}}^\top + b_{\text{res}}), \quad h_t = (1 - \alpha) h_{t-1} + \alpha \tilde{h}_t, \quad y_t = h_t W_{\text{out}}^\top + b_{\text{out}}
$$

### 2. Next-Generation Reservoir Computing (NG-RC)
Replaces stochastic neural reservoirs with deterministic polynomial delay taps:
$$
O_{\text{lin}} = [x_t, x_{t-1}], \quad O_{\text{nonlin}} = [x_{t, i} x_{t-1, j}], \quad O_{\text{total}} = [O_{\text{lin}}; O_{\text{nonlin}}]
$$

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

x_seq = weights["x_seq"]
alpha = float(weights["alpha"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, input shape: {list(x_seq.shape)}, leak alpha: {alpha}")

In [ ]:
def esn_forward(x_seq, w_in, w_res, b_res, alpha, w_out, b_out):
    B, T, D_in = x_seq.shape
    D_res = w_in.shape[0]
    h_t = torch.zeros(B, D_res, device=x_seq.device, dtype=x_seq.dtype)
    outs = []
    for t in range(T):
        xt = x_seq[:, t, :]
        tilde_h = torch.tanh(xt @ w_in.t() + h_t @ w_res.t() + b_res)
        h_t = (1.0 - alpha) * h_t + alpha * tilde_h
        outs.append((h_t @ w_out.t() + b_out).unsqueeze(1))
    return torch.cat(outs, dim=1), h_t

def ngrc_features(x_curr, x_prev):
    lin = torch.cat([x_curr, x_prev], dim=-1)
    quad = (x_curr.unsqueeze(-1) * x_prev.unsqueeze(-2)).flatten(start_dim=1)
    return torch.cat([lin, quad], dim=-1)

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
seq_out, final_h = esn_forward(
    x_seq, weights["w_in"], weights["w_res"], weights["b_res"],
    alpha, weights["w_out"], weights["b_out"]
)
err_s = float(torch.max(torch.abs(seq_out - weights["seq_out"])))
err_h = float(torch.max(torch.abs(final_h - weights["final_h"])))
assert err_s < 1e-6, f"ESN seq_out error: {err_s:.2e}"
assert err_h < 1e-6, f"ESN final_h error: {err_h:.2e}"

ngrc_feat = ngrc_features(weights["x_curr"], weights["x_prev"])
err_f = float(torch.max(torch.abs(ngrc_feat - weights["ngrc_feat"])))
assert err_f < 1e-6, f"NG-RC feature error: {err_f:.2e}"
print(f"Parity Verified: ESN Seq={err_s:.2e}, Final H={err_h:.2e}, NG-RC Features={err_f:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_esn = benchmark_fn(lambda: esn_forward(
    x_seq, weights["w_in"], weights["w_res"], weights["b_res"],
    alpha, weights["w_out"], weights["b_out"]
), iters=50, name="ESN Leaky Recurrence")
b_ngrc = benchmark_fn(lambda: ngrc_features(weights["x_curr"], weights["x_prev"]), iters=50, name="NG-RC Expansion")
print(format_benchmark_table([b_esn, b_ngrc]))

# Visualization: Reservoir Hidden Trajectory & NG-RC Polynomial Feature Map
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow(seq_out[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("ESN Readout Trajectory [T, D_out]")
ax[1].imshow(ngrc_feat.detach().numpy(), cmap="magma", aspect="auto")
ax[1].set_title("NG-RC Features [B, D_feat]")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Reservoir Spectral Radius Violation (rho >= 1.0)
# A reservoir with spectral radius > 1 loses fading memory and diverges exponentially!
w_unstable = torch.tensor([[1.3]])
h_state = torch.tensor([1.0])
for _ in range(15):
    h_state = torch.tanh(h_state * w_unstable) * 1.5  # Amplifying feedback

print(f"Chaotic/amplified state magnitude after 15 steps: {h_state.item():.2e}")
assert h_state.item() > 1.2, "Unstable spectral radius must violate contractive echo state property!"
print("Demonstrated requirement of contractive spectral radius rho < 1.0 in Reservoir Computing.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Zero BPTT Training:** Reservoir computing trains only linear readouts via ridge regression, avoiding vanishing gradients.
2. **Echo State Property:** Ensuring $\rho(W_{\text{res}}) < 1.0$ guarantees fading memory and numerical stability.
3. **Next-Generation Reservoir Computing:** Deterministic polynomial delay taps match or exceed stochastic reservoirs without hyperparameter tuning.

---
### Cross-Domain Generalisation Matrix
- **Physical Reservoir Computing:** Substrates like optical resonators, spintronic oscillators, and memristors compute reservoirs in analog physics.
- **Chaotic Dynamics Forecasting:** Accurately forecasts nonlinear chaotic attractors (Lorenz, Mackey-Glass).
- **Ultra-Low Power Edge Sensors:** Real-time biosignal anomaly detection on micropower hardware.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_reservoir_computing_echo_state_networks_lsm_and_ngrc.md](../../../markdown/05_architectures/03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.md)
- **PyTorch 2.x Module:** [07_reservoir_computing_echo_state_networks_lsm_and_ngrc.py](../../../pytorch2/05_architectures/03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.py)
- **Functional JAX Module:** [07_reservoir_computing_echo_state_networks_lsm_and_ngrc.py](../../../jax/05_architectures/03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.py)
- **Candle Rust Module:** [07_reservoir_computing_echo_state_networks_lsm_and_ngrc.rs](../../../candle/05_architectures/03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.rs)
- **Hardware Kernel:** [07_reservoir_computing_echo_state_networks_lsm_and_ngrc.rs](../../../kernels/05_architectures/03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.rs)
- **Universal Golden SafeTensors:** [07_reservoir_computing_echo_state_networks_lsm_and_ngrc.safetensors](../../../weights/05_architectures/03_recurrent_ssm_and_memory/07_reservoir_computing_echo_state_networks_lsm_and_ngrc.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 06. Memory Architectures NTM PKM PEER Infini and Titans](06_memory_architectures_ntm_pkm_peer_infini_and_titans.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 01. Attention Variants MHA MQA GQA MLA and DiffAttn](../04_transformers_and_heads/01_attention_variants_mha_mqa_gqa_mla_and_diffattn.ipynb) &rarr;